In [1]:
%pip install -q -U langchain langchain-openai openai ddgs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 6.3 MB/s eta 0:00:00


In [2]:
!pip uninstall -y duckduckgo-search
!pip install -U ddgs

## 1. Imports & model

In [3]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
)

In [5]:
from typing import Optional
from langchain.agents import create_agent
from langchain_core.tools import tool

from openai import OpenAI
from langchain_openai import ChatOpenAI

# MODEL_NAME = "llama3.1"
API_KEY ="sk-or-v1-tutimdi"
BASE_URL="https://openrouter.ai/api/v1"
MODEL_NAME="nvidia/nemotron-3-ultra-550b-a55b:free"


llm = ChatOpenAI(
    model=MODEL_NAME,
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=0.2,
)

## 2. Web search tool, search,...


Tool này chỉ dành cho trường hợp món ăn **mới lạ, hiếm, tên không rõ hoặc agent không đủ chắc chắn**.

In [6]:
from langchain_core.tools import tool
from ddgs import DDGS

@tool
def search_recipe_web(query: str) -> str:
    """
    Search web information about unfamiliar, rare, ambiguous,
    or regional dishes.
    """

    print(f"\n🔎 SEARCH QUERY: {query}")

    results = []

    with DDGS() as ddgs:
        for r in ddgs.text(
            query,
            region="vn-vi",
            safesearch="moderate",
            max_results=5,
        ):
            results.append(
                f"Title: {r.get('title', '')}\n"
                f"Content: {r.get('body', '')}\n"
                f"Source: {r.get('href', '')}"
            )

    if not results:
        return "Không tìm thấy kết quả phù hợp."

    return "\n\n".join(results)

## 3. Prompt điều khiển agent

Điểm quan trọng nhất: **món quen thuộc thì không được search**.

In [7]:
SEARCH_POLICY_PROMPT = """
QUY TẮC SỬ DỤNG TOOL SEARCH:

1. Trước tiên, hãy tự xác định xem món ăn có quen thuộc và bạn có đủ kiến thức
   để đưa ra một công thức hợp lý hay không.

2. Nếu món ăn quen thuộc và bạn nhận diện rõ:
   - KHÔNG được gọi tool `search_recipe_web`.
   - Trả lời trực tiếp bằng kiến thức sẵn có.

3. CHỈ được gọi `search_recipe_web` khi có ít nhất một trong các tình huống:
   - Món ăn mới lạ hoặc hiếm.
   - Tên món không rõ nghĩa / có thể bị viết sai.
   - Món quá đặc thù theo vùng miền hoặc văn hóa và bạn không chắc cách làm.
   - Bạn không đủ chắc chắn về nguyên liệu cốt lõi hoặc phương pháp nấu.

4. KHÔNG search chỉ để xác nhận lại một món quen thuộc.
5. Nếu đã search, chỉ lấy thông tin liên quan trực tiếp đến món ăn và tổng hợp lại;
   không chép nguyên văn kết quả search.
"""

SYSTEM_PROMPT = """
Bạn là Food Recipe Agent.

Bạn có tool search_recipe_web.

QUY TẮC SEARCH:
- Chỉ search khi món lạ, hiếm, mơ hồ hoặc bạn thật sự không biết.
- Món phổ biến thì trả lời trực tiếp.
- Nếu search, query phải tập trung vào FOOD / RECIPE.
- Không search các query quá chung chung.

Khi tạo search query, bắt buộc thêm các từ khóa phù hợp như:
"món ăn", "công thức", "recipe", "ẩm thực", "Vietnamese food".

Ví dụ user nói:
"Một món địa phương rất lạ mà tôi chỉ nhớ tên gần giống com trung"

Query tốt:
"món ăn Việt Nam cơm trứng đặc sản vùng miền công thức"

Không được lặp search nhiều lần với query gần giống nhau.
Nếu search lần đầu không có kết quả đáng tin cậy,
hãy nói rằng chưa xác định được món thay vì search vô hạn.
"""
{SEARCH_POLICY_PROMPT}
"""
YÊU CẦU ĐẦU RA:
- Tên món
- Khẩu phần
- Thời gian chuẩn bị và nấu
- Nguyên liệu kèm định lượng ước lượng
- Các bước thực hiện theo thứ tự
- Lưu ý/thay thế nguyên liệu nếu cần

NGUYÊN TẮC:
- Nếu người dùng chỉ mô tả món mà không biết tên, hãy suy luận tên món trước.
- Không bịa các chi tiết quá đặc thù nếu không chắc.
- Nếu tên món mơ hồ và search vẫn không đủ dữ liệu, hãy nói rõ phần nào chưa chắc.
- Trả lời bằng tiếng Việt, gọn, rõ, ưu tiên công thức có thể làm được tại nhà.
"""

'\nYÊU CẦU ĐẦU RA:\n- Tên món\n- Khẩu phần\n- Thời gian chuẩn bị và nấu\n- Nguyên liệu kèm định lượng ước lượng\n- Các bước thực hiện theo thứ tự\n- Lưu ý/thay thế nguyên liệu nếu cần\n\nNGUYÊN TẮC:\n- Nếu người dùng chỉ mô tả món mà không biết tên, hãy suy luận tên món trước.\n- Không bịa các chi tiết quá đặc thù nếu không chắc.\n- Nếu tên món mơ hồ và search vẫn không đủ dữ liệu, hãy nói rõ phần nào chưa chắc.\n- Trả lời bằng tiếng Việt, gọn, rõ, ưu tiên công thức có thể làm được tại nhà.\n'

## 4. Tạo agent

In [8]:
tools = [search_recipe_web,]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

## 5. Hàm hỏi công thức

In [9]:
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage


def run_agent_with_trace(agent, user_input: str):
    print("=" * 70)
    print("👤 USER")
    print(user_input)
    print("=" * 70)

    final_answer = None

    for chunk in agent.stream(
        {
            "messages": [
                {
                    "role": "user",
                    "content": user_input
                }
            ]
        },
        stream_mode="values"
    ):
        messages = chunk.get("messages", [])

        if not messages:
            continue

        msg = messages[-1]

        # ============================
        # MODEL / AGENT
        # ============================
        if isinstance(msg, AIMessage):

            # Agent quyết định gọi tool
            if getattr(msg, "tool_calls", None):
                for call in msg.tool_calls:
                    print("\n🤖 AGENT DECISION")
                    print("→ Gọi tool:", call["name"])
                    print("→ Arguments:", call["args"])

            # Agent trả text
            elif msg.content:
                print("\n🤖 AGENT")
                print(msg.content)
                final_answer = msg.content

        # ============================
        # TOOL RESULT
        # ============================
        elif isinstance(msg, ToolMessage):
            print("\n🔧 TOOL RESULT")
            print("Tool:", msg.name)
            print("Result:")
            print(msg.content)

    print("\n" + "=" * 70)
    print("✅ FINAL ANSWER")
    print(final_answer)
    print("=" * 70)

    return final_answer

## 6. Demo

Với món quen thuộc như **cơm rang trứng**, agent nên trả lời trực tiếp và không search.

In [12]:
recipe = run_agent_with_trace(agent,"Cơm rang trứng")
print(recipe)

👤 USER
Cơm rang trứng

🤖 AGENT
**Cơm rang trứng** (Egg Fried Rice) là món ăn phổ biến, dễ làm, thường dùng cơm nguội (cơm đêm) để rang cho hạt tách biệt, không bị nát.

---

### 📝 Nguyên liệu (2–3 người)

| Nguyên liệu | Số lượng |
|-------------|----------|
| Cơm trắng (nên dùng cơm nguội, để tủ lạnh 1 đêm) | 2–3 bát |
| Trứng gà | 2–3 quả |
| Hành tây / hành tím | ½ quả, băm nhỏ |
| Cà rốt | ½ quả, cắt hạt lựu |
| Đậu Hà Lan (đóng hộp hoặc tươi luộc chín) | ½ chén |
| Xúc xích / thịt nguội / tôm khô (tuỳ thích) | 100 g, cắt nhỏ |
| Hành lá | 2–3 cây, thái nhỏ (phần xanh) |
| Dầu ăn | 2–3 thìa canh |
| Nước mắm | 1–2 thìa canh |
| Hạt nêm / bột ngọt | ½ thìa cà phê |
| Tiêu đen | Vừa đủ |
| Tỏi (tuỳ chọn) | 2 tép, băm nhỏ |

---

### 👩‍🍳 Cách làm

1. **Chuẩn bị cơm**  
   - Cơm nguội lấy ra, dùng tay hoặc thìa bẻ nhẹ cho hạt tách biệt, không bị vón cục.

2. **Chiên trứng**  
   - Đập trứng vào tô, đánh tan với chút tiêu, 1 chút nước mắm.  
   - Bắc chảo nóng, cho 1 thìa dầu, đổ trứng 

### Thử với món lạ / tên mơ hồ

Đổi chuỗi bên dưới thành một món bạn không quen. Khi agent không đủ chắc chắn, nó mới được phép gọi tool search.

In [11]:
# Ví dụ:
recipe = run_agent_with_trace(agent,"Một món địa phương rất lạ mà tôi chỉ nhớ tên gần giống com trung")
print(recipe)

👤 USER
Một món địa phương rất lạ mà tôi chỉ nhớ tên gần giống com trung

🤖 AGENT DECISION
→ Gọi tool: search_recipe_web
→ Arguments: {'query': 'món ăn Việt Nam đặc sản vùng miền tên gần giống cơm trứng công thức'}

🔎 SEARCH QUERY: món ăn Việt Nam đặc sản vùng miền tên gần giống cơm trứng công thức

🔧 TOOL RESULT
Tool: search_recipe_web
Result:
Title: 40 Đặc Sản, Món Ngon Miền Nam Mà Bạn Nhất Định Phải Thử
Content: 17 thg 11, 2022 · Bánh cống Cần Thơ - nghe cái tên cũng thật lạ nhưng nguyên liệu món ăn lại rất quen thuộc, bao gồm bột gạo, bột nếp, đậu xanh, tôm, thịt,...
Source: https://vincom.com.vn/trend/mon-ngon-mien-nam

Title: TOP 20 Đặc sản miền Trung nổi tiếng | Khám phá ẩm thực #3
Content: Bún bò Huế có hương vị rất riêng, không thể “lẫn” với những loại bún khác bởi vị ngon đặc trưng của mắm ruốc, hương sả, khhi thưởng thức sẽ có vị ngọt dịu dàng, ...
Source: https://banhkhome.com/top-20-dac-san-mien-trung-noi-tieng

Title: 15 món đặc sản miền Trung đáng nhớ không thể bỏ qua - P